# Attention-UNet++ on MoNuSAC (Kaggle T4)
1. Upload this folder as a Kaggle **Dataset** (e.g. `nucseg-code`) and add the MoNuSAC dataset to the notebook.
2. Accelerator: **GPU T4 x1** (one GPU is used). Internet **on** if you want ImageNet weights (else see README).
3. Run top to bottom.

In [ ]:
import os, shutil, sys
SRC = next((p for p in ['/kaggle/input/nucseg-code/nucseg_monusac', '/kaggle/input/nucseg-code'] if os.path.isdir(p + '/nucseg')), None)
if SRC and not os.path.isdir('/kaggle/working/nucseg_monusac'):
    shutil.copytree(SRC, '/kaggle/working/nucseg_monusac')
PROJ = '/kaggle/working/nucseg_monusac' if SRC else os.getcwd()
os.chdir(PROJ); sys.path.insert(0, PROJ)
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

In [ ]:
from nucseg.config import Config
cfg = Config(
    data_root='auto',        # or e.g. '/kaggle/input/monusac-2020'
    encoder='resnet34',      # 'scratch' = your original from-scratch residual encoder (for the ablation)
    pretrained=True,         # needs Internet ON, or set encoder_weights_path
    epochs=60, batch_size=12, crop=256,
    time_budget_min=500,     # stop cleanly before the session limit; resume with resume=True
).resolve()
cfg.to_dict()

## 1. EDA (tables + 8 figures)

In [ ]:
from nucseg.engine import prepare
from nucseg.eda import run_eda
data = prepare(cfg)          # parses XML -> cached .npz (first run only), patient-level splits
img_df, nuc_df = run_eda(cfg, data)

## 2. Train

In [ ]:
from nucseg.engine import train
hist = train(cfg, data)

## 3. Evaluate (post-processing tuned on val, applied to test)

In [ ]:
from nucseg.engine import evaluate
res_val = evaluate(cfg, 'val', data=data)
res_test = evaluate(cfg, 'test', data=data) if 'test' in data else None

## 4. Package outputs

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/monusac_run', 'zip', cfg.out_dir)
print(sorted(os.listdir(cfg.out_dir + '/figures')))